# 1.2 상품주문

고객이 말한 메뉴를 주문 초안으로 정리하고, 실제 메뉴와 필수 정보를 확인해 고객 승인 후 모의 접수합니다.

- **입력:** 고객 문장과 주문 상태가 담긴 `SessionState`
- **처리:** 주문 추출 → 메뉴·필수 정보 확인 → 고객 확인 → 모의 접수
- **출력:** 확인 질문 또는 테스트 주문 결과가 담긴 `SessionState`

예: “아이스 아메리카노 두 잔”을 입력하면 빠진 옵션과 이용 방식을 먼저 확인합니다. 프로젝트 루트에서 실행하며 실제 모델 호출에는 `OPENAI_API_KEY`가 필요합니다.

```mermaid
flowchart LR
    input["Input<br/>SessionState"] --> draft["주문 초안 생성<br/>generate_order_draft(state)"]
    draft --> check["메뉴·정보 확인<br/>validate_order(state)"]
    check --> handle["고객 확인·모의 접수<br/>handle_order(state)"]
    handle --> output["Output<br/>SessionState"]
    classDef io fill:#eeeeee,stroke:#999999,color:#333333;
    class input,output io;
```

## 1. 스키마 설정

공통 주문·세션 스키마를 불러옵니다.

In [1]:
# 1. 경로 준비: 프로젝트 루트를 공통 스키마를 찾는 경로에 추가합니다.
import sys
import os
sys.path.append(os.getcwd())

# 2. 스키마 확인: 주문 초안과 고객 상태에 필요한 자료형을 불러옵니다.
from schemas.product_order import OrderDraft
from schemas.session_state import CustomerInput, IntentResult, OrderState, SessionState
print("스키마:", OrderDraft.__name__, SessionState.__name__)
print("주문 항목:", list(OrderState.model_fields))

스키마: OrderDraft SessionState
주문 항목: ['items', 'order_type', 'payment_method', 'total_price', 'order_id', 'confirmation_status', 'needs_clarification', 'question']


## 2. 모델 준비

환경변수로 OpenAI 클라이언트와 모델을 준비합니다.

In [2]:
# 1. 모델 준비: 환경변수에서 키와 모델명을 읽어 클라이언트를 만듭니다.
from openai import OpenAI
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
model = os.getenv("OPENAI_MODEL", "gpt-5-nano").strip() or "gpt-5-nano"
print("설정한 모델:", model)

설정한 모델: gpt-5-nano


## 3. 관련 프롬프트 불러오기

말투와 주문 추출 지침을 읽고, 실제 메뉴와 옵션을 확인합니다.

In [3]:
# 1. 지침 읽기: 공통 말투와 주문 추출 지침을 읽습니다.
from pathlib import Path
import json
persona = Path("prompts/persona.md").read_text(encoding="utf-8")
order_prompt = Path("prompts/order_extraction.md").read_text(encoding="utf-8")
print(persona)
print(order_prompt)

# 2. 자료 읽기: 실제 메뉴·옵션을 읽어 검증과 모델 참고에 사용합니다.
menu = json.loads(Path("data/menu.json").read_text(encoding="utf-8"))
options = json.loads(Path("data/order_options.json").read_text(encoding="utf-8"))
print("메뉴 개수:", len(menu))
print("옵션:", list(options))

# 챗봇 성격과 말투

당신은 모찌카페의 친절하고 차분한 안내 도우미다.

- 고객을 존중하며 자연스러운 한국어 존댓말을 사용한다.
- 고객이 궁금해하거나 요청한 내용에 집중하여 이해하기 쉽게 답한다.
- 답변은 필요한 내용 위주로 간결하게 작성한다.
- 과장된 표현이나 불필요한 감탄, 반복적인 인사는 피한다.
- 확인되지 않은 내용을 지어내지 않고, 필요한 경우 고객에게 확인한다.

# 역할

당신은 모찌카페 고객의 주문 요청을 고객 확인 전 JSON 초안으로 만든다. 필요한 내용이 없으면 하나씩 질문하며, 실제 주문을 접수하지 않는다.

## 입력

- 입력은 `SessionState`다.
- `input.text`: 고객의 현재 주문 요청 또는 수정 문장
- `order`: 세션의 기존 주문 초안과 확인 상태. 첫 요청이면 null
- system 메시지의 메뉴·옵션 참고자료: 실제 메뉴 목록과 상품별 판매 여부·온도·옵션 정보
- 이전 대화: 앞선 고객 요청과 챗봇의 확인 질문

## 처리 기준

- 기존 `order`가 있으면 항상 그 주문을 수정한 최종 초안을 반환한다. 새 문장이 들어올 때마다 상품을 기존 목록 뒤에 누적하지 않는다. 현재 `input.text`의 명시적인 값은 이전 `order`와 대화보다 우선한다.
- 기존 상품을 다시 말하면 그 상품의 수량·온도·옵션을 최신 값으로 덮어쓴다. 예를 들어 기존 아메리카노 2잔에 “아메리카노 한 잔”이라고 하면 1잔으로 수정하며, 3잔으로 더하거나 이전 2잔을 유지하지 않는다.
- “추가로”, “한 잔 더”, “이것도 추가”처럼 추가 의사를 명시한 경우에만 상품이나 수량을 더한다. 추가 의사가 없는데 다른 상품을 말하면 기존 상품을 바꾸려는 요청으로 처리하며, 어느 상품을 바꿀지 불분명하면 누적하지 말고 질문한다.
- 메뉴를 다시 선택해 달라는 안내 뒤에 고객이 주문 상품들을 제시하면, 거절된 상품이나 이름을 특정하지 못한 항목을 새 선택으로 교체한다. 전체 주문 목록을 다시 제시한 경우에는 그 

## 4. 사용자 input 준비

고객 주문 한 문장을 상태에 담습니다.

In [4]:
# 1. 입력 준비: 온도와 수량은 있지만 다른 정보는 확인이 필요한 주문입니다.
state = SessionState(input=CustomerInput(text="아이스 아메리카노 두 잔 주세요"), intent=IntentResult(intent="product_order"))
print("고객 입력:", state.input.text)
print("의도:", state.intent.intent)

고객 입력: 아이스 아메리카노 두 잔 주세요
의도: product_order


## 5. 함수 만들기

주문을 추출하고, 실제 자료로 확인한 뒤 고객 응답을 처리합니다.

In [5]:
def generate_order_draft(state: SessionState):
    """고객 주문과 이전 초안을 모델에 보내 주문 초안을 상태에 담아 돌려줍니다."""
    # 1. 입력 준비: 메뉴 이름과 선택지를 지침에 붙여 현재 상태와 함께 전달합니다.
    reference = [{key: item[key] for key in ("name", "aliases", "temperatures", "available")} for item in menu]
    system = f"{persona}\n\n{order_prompt}\n\n## 메뉴 참고자료\n{json.dumps(reference, ensure_ascii=False)}\n\n## 옵션 참고자료\n{json.dumps(options, ensure_ascii=False)}"

    ## 참고자료가 담긴 지침 뒤에 이전 대화와 이번 상태를 붙입니다.
    messages = [{"role": "system", "content": system}]
    messages.extend(message.model_dump(mode="json") for message in state.history)
    messages.append({"role": "user", "content": state.model_dump_json(exclude={"history", "customer"})})

    # 2. 모델 호출: 고객이 말한 주문 내용을 정해진 형식으로 추출합니다.
    response = client.responses.parse(model=model, input=messages, text_format=OrderDraft)
    if response.output_parsed is None:
        raise ValueError("주문 초안을 받지 못했습니다.")

    # 3. 상태 저장: 금액은 추측하지 않고 실제 메뉴 확인 전 상태로 둡니다.
    state.order = OrderState(**response.output_parsed.model_dump(mode="json"))
    state.order.total_price = None

    return state


In [6]:
def validate_order(state: SessionState):
    """주문 초안을 메뉴·옵션과 비교해 누락 여부와 확인 질문을 상태에 담아 돌려줍니다."""
    # 1. 주문 확인: 기존 질문이 있거나 주문 항목이 없으면 고객에게 확인합니다.
    order = state.order
    if order is None or not order.items:
        state.answer = {"answer": order.question if order and order.question else "어떤 메뉴를 주문하시겠어요?"}

        return state
    if order.needs_clarification:
        state.answer = {"answer": order.question or "주문 내용을 다시 알려주세요."}

        return state

    # 2. 메뉴 확인: 실제 이름·별칭과 판매 여부를 확인하고 기본 가격을 채웁니다.
    missing = []
    for item in order.items:
        item.product_id = None
        item.price = None
        matches = [entry for entry in menu if item.product_name in [entry["name"], *entry.get("aliases", [])]]
        if not matches or not matches[0]["available"]:
            state.answer = {"answer": f"{item.product_name or '해당 메뉴'}는 주문할 수 없습니다. 다른 메뉴를 알려주세요."}

            return state

        ## 판매 가능한 메뉴의 실제 ID와 이름, 기본 가격을 채웁니다.
        selected = matches[0]
        item.product_id = selected["menu_id"]
        item.product_name = selected["name"]
        item.price = selected["price"]

        ## 수량과 온도에서 빠지거나 허용되지 않는 값을 확인합니다.
        if item.quantity is None or item.quantity <= 0 or not float(item.quantity).is_integer():
            missing.append(f"{item.product_name} 수량")
        if selected["temperatures"] and item.temperature not in selected["temperatures"]:
            missing.append(f"{item.product_name} 온도")
        elif not selected["temperatures"] and item.temperature is not None:
            state.answer = {"answer": f"{item.product_name}은 온도 선택이 없습니다."}

            return state

        ## 옵션 선택 여부와 중복·조합·메뉴별 적용 가능 여부를 확인합니다.
        if item.options is None:
            missing.append(f"{item.product_name} 옵션 여부")
        else:
            if len(item.options) != len(set(item.options)) or ("두유로 변경" in item.options and "오트 음료로 변경" in item.options):
                state.answer = {"answer": "중복되거나 함께 적용할 수 없는 옵션이 있습니다."}

                return state
            for option in item.options:
                if option not in options or item.product_id not in options[option]["menu_ids"]:
                    state.answer = {"answer": f"{item.product_name}에는 {option} 옵션을 적용할 수 없습니다."}

                    return state

    # 3. 필수 정보 확인: 이용과 결제 정보를 포함해 빠진 내용을 질문합니다.
    if order.order_type is None:
        missing.append("매장 이용 또는 포장")
    if order.payment_method is None:
        missing.append("결제 방식")

    ## 누락된 정보가 있으면 질문하고, 모두 있으면 최종 확인을 요청합니다.
    if missing:
        state.answer = {"answer": "주문을 위해 " + ", ".join(missing) + "을 알려주세요."}
    else:
        order.confirmation_status = "pending"
        summary = ", ".join(f"{item.product_name} {int(item.quantity)}개 ({item.temperature or '온도 없음'}, 옵션: {', '.join(item.options) or '없음'})" for item in order.items)
        state.answer = {"answer": f"{summary} / {order.order_type} / {order.payment_method} 맞나요? yes / no로 답해 주세요."}

    return state


In [7]:
def handle_order(state: SessionState):
    """고객 확인 답변과 주문 상태를 받아 승인된 주문만 모의 접수하고 결과를 돌려줍니다."""
    # 1. 확인 상태 검사: 확인 대기 주문과 명시적인 승인 답변이 있어야 접수합니다.
    if state.order is None or state.order.confirmation_status != "pending":
        state.answer = {"answer": "확인할 주문이 없습니다."}

        return state
    if state.input.text.strip().lower() != "yes":
        state.order.confirmation_status = "awaiting_revision"
        state.answer = {"answer": "수정할 주문 내용을 알려주세요."}

        return state

    # 2. 재검증: 승인 시 메뉴와 필수 정보를 다시 확인합니다.
    state = validate_order(state)
    if state.order.confirmation_status != "pending" or not state.answer["answer"].endswith("yes / no로 답해 주세요."):

        return state

    # 3. 모의 접수: 실제 주문 저장과 결제 없이 메뉴·옵션 가격으로 금액을 계산합니다.
    from uuid import uuid4
    total = sum((item.price + sum(options[name]["price"] for name in item.options)) * int(item.quantity) for item in state.order.items)

    ## 계산한 금액과 테스트 주문번호를 확정 결과에 담습니다.
    state.order.order_id = f"test-{uuid4().hex[:12]}"
    state.order.total_price = total
    state.order.confirmation_status = "confirmed"
    state.answer = {"answer": f"테스트 주문번호 {state.order.order_id}, 결제금액 {total:,}원입니다."}

    return state

## 6. 연결 테스트

모델을 실제로 호출합니다. 테스트 접수는 메모리 상태만 변경하며 실제 주문과 결제는 하지 않습니다.

### 테스트 1 - 부족한 정보 확인

In [8]:
# 1. 입력 준비: 온도와 수량만 말한 주문을 준비합니다.
state = SessionState(input=CustomerInput(text="아이스 아메리카노 두 잔 주세요"), intent=IntentResult(intent="product_order"))
print("Input:", state.input.text)

# 2. 함수 호출: 고객 주문을 추출하고 실제 메뉴와 필요한 정보를 확인합니다.
print("\nFunction: generate_order_draft → validate_order")
state = generate_order_draft(state)
print("주문 초안:", state.order.model_dump_json(indent=2))
state = validate_order(state)

# 3. 결과 확인: 빠진 정보를 묻는 답변을 보여줍니다.
print("\nOutput:", state.answer["answer"])

Input: 아이스 아메리카노 두 잔 주세요

Function: generate_order_draft → validate_order


주문 초안: {
  "items": [
    {
      "product_id": null,
      "product_name": "아메리카노",
      "quantity": 2.0,
      "temperature": "ICE",
      "options": null,
      "price": null
    }
  ],
  "order_type": null,
  "payment_method": null,
  "total_price": null,
  "order_id": null,
  "confirmation_status": "pending",
  "needs_clarification": false,
  "question": null
}

Output: 주문을 위해 아메리카노 옵션 여부, 매장 이용 또는 포장, 결제 방식을 알려주세요.


### 테스트 2 - 주문 확인과 모의 접수

완전한 주문을 새로 만들고 다음 턴에 승인합니다.

In [9]:
# 1. 입력 준비: 옵션·이용·결제 방식까지 밝힌 주문을 준비합니다.
state = SessionState(input=CustomerInput(text="아이스 아메리카노 두 잔, 옵션 없이 포장하고 카드로 결제할게요."), intent=IntentResult(intent="product_order"))
print("Input:", state.input.text)

# 2. 함수 호출: 주문을 구조화하고 고객 확인 질문을 만듭니다.
print("\nFunction: generate_order_draft → validate_order")
state = generate_order_draft(state)
state = validate_order(state)
print("확인 질문:", state.answer["answer"])

# 3. 다음 턴: 확인 대기 상태일 때 고객의 yes를 같은 상태에 담아 접수합니다.
if state.order.confirmation_status == "pending" and state.answer["answer"].endswith("yes / no로 답해 주세요."):
    state.input = CustomerInput(text="yes")
    print("\nInput:", state.input.text)
    print("Function: handle_order")
    state = handle_order(state)

# 4. 결과 확인: 실제 접수 대신 테스트 주문 결과를 보여줍니다.
print("\nOutput:", state.answer["answer"])
print("주문 상태:", state.order.confirmation_status if state.order else None)

Input: 아이스 아메리카노 두 잔, 옵션 없이 포장하고 카드로 결제할게요.

Function: generate_order_draft → validate_order


확인 질문: 아메리카노 2개 (ICE, 옵션: 없음) / 포장 / 카드 맞나요? yes / no로 답해 주세요.

Input: yes
Function: handle_order

Output: 테스트 주문번호 test-ae9fa13547b9, 결제금액 9,000원입니다.
주문 상태: confirmed
